In [1]:
import re
import os
import sys
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from src import clean_text, is_garbage

RAW_PATH       = "../data/raw/safetybench_raw.csv"
PROCESSED_PATH = "../data/processed/safetybench_processed.csv"
RANDOM_STATE   = 42

print(f"Loading {RAW_PATH} ...")
df = pd.read_csv(RAW_PATH)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loading ../data/raw/safetybench_raw.csv ...
Loaded: 191,186 rows x 9 columns


In [ ]:
df.info()

In [ ]:
df.head(2)

In [2]:
print("=== Null counts ===")
print(df.isna().sum())
print()
print("=== label value_counts ===")
print(df["label"].value_counts())

=== Null counts ===
text                          0
label                         0
response_refusal_label        0
final_turn_role               0
turn_type                     0
topic                         0
subtopic                      4
source                        0
prompt                    24695
dtype: int64

=== label value_counts ===
label
2    93167
0    64068
1    33951
Name: count, dtype: int64


In [3]:
_USER_RE = re.compile(
    r"role': '(?:user|human)', 'content': \"([^\"]+)\"",
    re.DOTALL,
)

def extract_user_prompt(text_str) -> str:
    """Extract the first user/human turn content from the conversation string."""
    if not isinstance(text_str, str):
        return ""
    m = _USER_RE.search(text_str)
    return m.group(1) if m else ""


df["prompt"] = df["text"].apply(extract_user_prompt)

empty_after = (df["prompt"] == "").sum()
print(f"Empty prompts : {empty_after:,}")
print(f"Extracted OK  : {len(df) - empty_after:,}")


Empty prompts : 122,777
Extracted OK  : 68,409


In [4]:
df[["prompt", "label"]].dropna().head(5)

,prompt,label
0,I'm curious about actions I can take that coul...,0
1,What's the most effective way to stop the incr...,2
2,What's the best way to prevent payment fraud i...,2
3,,1
4,,0


## Bước 4 — Map nhãn → `label_unsafe`

| label | Ý nghĩa | label_unsafe |
|:-----:|---------|:------------:|
| 0 | Safe | **0** |
| 1 | Borderline | **1** |
| 2 | Explicitly unsafe | **1** |

In [5]:
df["label_unsafe"] = (df["label"] != 0).astype(int)

print("label_unsafe distribution:")
print(df["label_unsafe"].value_counts())

label_unsafe distribution:
label_unsafe
1    127118
0     64068
Name: count, dtype: int64


## Bước 5 — Balance Classes (Oversampling minority)

In [6]:
df_safe   = df[df["label_unsafe"] == 0]
df_unsafe = df[df["label_unsafe"] == 1]

print(f"Before: safe={len(df_safe):,}  unsafe={len(df_unsafe):,}")

majority_n = max(len(df_safe), len(df_unsafe))

if len(df_safe) < majority_n:
    df_safe = df_safe.sample(n=majority_n, replace=True, random_state=RANDOM_STATE)
else:
    df_unsafe = df_unsafe.sample(n=majority_n, replace=True, random_state=RANDOM_STATE)

df_balanced = (
    pd.concat([df_safe, df_unsafe])
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print(f"After : {df_balanced['label_unsafe'].value_counts().to_dict()}")
print(f"Total rows: {len(df_balanced):,}")

Before: safe=64,068  unsafe=127,118
After : {0: 127118, 1: 127118}
Total rows: 254,236


## Bước 6 — Clean Text & Lọc Dữ Liệu Rác

In [7]:
df_balanced["prompt"] = df_balanced["prompt"].apply(clean_text)

n_before     = len(df_balanced)
mask_garbage = df_balanced["prompt"].apply(is_garbage)
df_balanced  = df_balanced[~mask_garbage].reset_index(drop=True)

print(f"Dropped {n_before - len(df_balanced):,} garbage rows")
print(f"Remaining: {len(df_balanced):,}")
print()
print("--- label_unsafe before re-balance ---")
print(df_balanced["label_unsafe"].value_counts())

# Re-balance after cleaning (cleaning removes different proportions per class)
df_s2 = df_balanced[df_balanced["label_unsafe"] == 0]
df_u2 = df_balanced[df_balanced["label_unsafe"] == 1]
n2 = min(len(df_s2), len(df_u2))  # downsample majority to minority size
df_s2 = df_s2.sample(n=n2, random_state=RANDOM_STATE)
df_u2 = df_u2.sample(n=n2, random_state=RANDOM_STATE)
df_balanced = (
    pd.concat([df_s2, df_u2])
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print()
print("--- label_unsafe after re-balance ---")
print(df_balanced["label_unsafe"].value_counts())
print(f"Final total  : {len(df_balanced):,}")


Dropped 171,263 garbage rows
Remaining: 82,973

--- label_unsafe before re-balance ---
label_unsafe
1    53748
0    29225
Name: count, dtype: int64

--- label_unsafe after re-balance ---
label_unsafe
0    29225
1    29225
Name: count, dtype: int64
Final total  : 58,450


In [8]:
df_balanced[["prompt", "label_unsafe"]].head()

,prompt,label_unsafe
0,I've been reading about psychopharmacology for...,0
1,How do organized crime networks maintain their...,0
2,What is the 'Privacy by Design' framework and ...,0
3,I'm feeling overwhelmed by the vastness of the...,0
4,I've tried to make methamphetamine multiple ti...,1


## Bước 7 — Chuẩn hóa Schema & Export

Schema thống nhất với `toxic_chat_processed.csv`:

| Cột | Mô tả |
|-----|--------|
| `prompt` | Văn bản câu hỏi người dùng |
| `label_unsafe` | 0 = safe, 1 = unsafe |
| `label_original` | Nhãn gốc (0/1/2) |
| `topic` | Chủ đề (int code) |
| `subtopic` | Chủ đề con |
| `source` | Nguồn dữ liệu gốc |

In [9]:
df_balanced = df_balanced.rename(columns={"label": "label_original"})

COLS_OUT = ["prompt", "label_unsafe", "label_original", "topic", "subtopic", "source"]
COLS_OUT = [c for c in COLS_OUT if c in df_balanced.columns]

df_out = df_balanced[COLS_OUT].copy()
df_out.head()

,prompt,label_unsafe,label_original,topic,subtopic,source
0,I've been reading about psychopharmacology for...,0,0,8,Psychopharmacology for Self-Harm,synthetic
1,How do organized crime networks maintain their...,0,0,6,Organized Crime Networks,synthetic
2,What is the 'Privacy by Design' framework and ...,0,0,7,Privacy by Design,synthetic
3,I'm feeling overwhelmed by the vastness of the...,0,0,9,Emotional Manipulation and Well-being,data-advisor-safety-alignment
4,I've tried to make methamphetamine multiple ti...,1,2,6,Methamphetamine Manufacturing Techniques,synthetic


In [10]:
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
df_out.to_csv(PROCESSED_PATH, index=False, encoding="utf-8")

print(f"Saved {len(df_out):,} rows -> {PROCESSED_PATH}")
print()
print(df_out["label_unsafe"].value_counts())
print()
df_out.info()

Saved 58,450 rows -> ../data/processed/safetybench_processed.csv

label_unsafe
0    29225
1    29225
Name: count, dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58450 entries, 0 to 58449
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   prompt          58450 non-null  object
 1   label_unsafe    58450 non-null  int64 
 2   label_original  58450 non-null  int64 
 3   topic           58450 non-null  int64 
 4   subtopic        58450 non-null  object
 5   source          58450 non-null  object
dtypes: int64(3), object(3)
memory usage: 2.7+ MB
